<img style="float: center;" src='https://github.com/spacetelescope/jwst-pipeline-notebooks/raw/main/_static/stsci_header.png' alt="stsci_logo" width="900px"/>

# RW-DDT Checkpoint Analysis Template Notebook

**Authors**: Taylor James Bell (ESA/AURA for STScI)<br>
**Last Updated**: January 29, 2026<br>
**Eureka! Pipeline Version**: https://github.com/kevin218/Eureka/tree/tjb_rwddt

Note that some additional contextual information can be found in `README_DeepDive.md`

**Purpose**:<br/>

This notebook allows you to simultaneously fit two or more eclipse observations, using Eureka!'s Stage 5 code. The primary objective of this notebook is to complete the RWDDT CIT's JWST Data Analysis Team's work on Checkpoints 1 & 2 for each RWDDT target.

For Checkpoint 1, the primary focus is to attempt to obtain a more precise estimate of the mid-eclipse time in the hopes of enabling more efficient observations of future eclipses. This will not necessarily be possible, as our checkpoint 1 strategy is set such that the eclipse be detectable at ≥3σ in the "bare-rock" case and assuming that the eccentricity is small enough that we've captured the eclipse with our Checkpoint 1 eclipses; therefore, a non-detection may mean a fainter eclipse signal (potentially a non-zero albedo or an atmosphere), or it may mean that the planet isn't on a circular orbit and we need to keep expanding our observing windows. However, by simultaneously fitting two or more eclipses at the same time, we will have a better chance at constraining the mid-eclipse time than for each of our signle-eclipse fits.

Information about Checkpoint 2 will be added once that checkpoint is reached for the first time.

**Data**:<br/>
This notebook assumes the Stage 4 `_lc.hdf5` files from all past eclipses have already been produced.

---

## Table of Contents
- [0. Importing the required components](#0.-Importing-the-required-components)
  - [0.1 Define your eventlabel and top directory](#0.1-Define-your-eventlabel-and-top-directory)
- [1. Stage 5 - Fitting the lightcurves](#5.-Stage-5---Fitting-the-lightcurves)

---

## 0. Importing the required components

There should be no need to change any of this

In [ ]:
# Importing a bunch of Eureka! components
import eureka
import eureka.S5_lightcurve_fitting.s5_fit as s5

# Set up some parameters to make plots look nicer. You can set usetex=True if you have LaTeX installed
eureka.lib.plots.set_rc(style='eureka', usetex=False, filetype='.png')

# Some imports to interact with outputs within the Jupyter notebook
import os
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

### 0.1 Define your eventlabel and top directory

Next, we need to choose a short, meaningful label (without spaces) that describes the data we're currently working on. This eventlabel will determine will give nicknames to all your output folders and files.

We also need to tell the notebook where all our data is going to be stored

In [ ]:
# Specify here the top directory that will contain all ingested and output files
topdir = '/home/rwddt' ## <- ENTER YOUR TOPDIR HERE (leave as-is if using Docker)

# Specify your analysis name here (leave as-is if using Docker)
analysis_name = 'analysis'

# Paths to previous Stage 4 analysis folders with respect to topdir
# NOTE: Must specify the specific ap#_bg#_# folder, otherwise the joint fit will just use the newest one
# NOTE: Paths must be to the ap#_bg#_# folder and not the _LCData.h5 file
inputdirlist = ['/Path/To/Eclipse01/Stage4/S4_.../ap#_bg#_#',
                '/Path/To/Eclipse02/Stage4/S4_.../ap#_bg#_#']

---

## 1. Stage 5 - Fitting the lightcurves

### 1.0 Quick sanity check that the intputdirlist folders exist

In [ ]:
errored = False
for i, inputdir in enumerate(inputdirlist):
    path = os.path.join(topdir, inputdir)
    found = os.path.isdir(path)
    if not found:
        errored = True
        raise AssertionError(f'Could not find the folder: {os.path.join(topdir, inputdir)}')
    elif i == 0:
        # Get the eventlabel from inputdirlist[0], which is required by Eureka! for joint fits
        paths = glob(os.path.join(path, '*SpecData.h5'))
        if len(paths)==0:
            paths = glob(os.path.join(path, '*Meta_Save.dat'))
        if len(paths)==0:
            raise AssertionError(f'Unable to find a Eureka!-compatible metadata file in {path}')
        s4_path = paths[0]
        s4_meta = eureka.lib.manageevent.loadevent(s4_path)
        eventlabel = s4_meta.eventlabel
if not errored:
    print('All specified paths exist')

### 1.1 Setting up the Stage 5 ECF

For the checkpoint analyses, the main thing you may want to experiment with changing is `run_myfuncs` (i.e., turning on/off each of the `expramp`, `xpos`, `ypos`, `xwidth`, `ywidth`, and/or `GP` models).

You may also want to change the `manual_clip` setting, which removes specified integrations from the fit. For example, a value of `[[None,450]]` will remove the first 450 integrations from all eclipse observations, while a value of `[[[None,450],], [[None,500],]]` removes 450 integrations from the first eclipse observation and 500 integrations from the second. You can also specify any other bad points (e.g., a small flare event or something) in the middle of the observations. For example, if you want to remove the first 450 integrations from eclipse 1, the first 500 integrations from eclipse 2, and integrations 1000 to 1010 from eclipse 2, you'd specify `[[[None,450],], [[None,500], [999,1010]]]`.

In [ ]:
s5_ecf_contents = f"""# Eureka! Control File for Stage 5: Lightcurve Fitting

# Stage 5 Documentation: https://eurekadocs.readthedocs.io/en/latest/ecf.html#stage-5

ncpu            16    # The number of CPU threads to use when running emcee or dynesty in parallel

multwhite       True  # This parameter tells Eureka! that you're fitting multiple observations at the same time

allapers        False # Run S5 on all of the apertures considered in S4? Otherwise will use newest output in the inputdir

fit_par         ./S5_{eventlabel}.epf
fit_method      [dynesty]
run_myfuncs     [batman_ecl, polynomial, expramp, xpos, ypos, xwidth, ywidth]  # By default, the notebook doesn't include a GP and you'd need to add GP here to include it

#GP inputs
kernel_inputs   ['time']  # options: time
kernel_class    ['Matern32']  # options: ExpSquared, Matern32, Exp, RationalQuadratic for george, Matern32 for celerite (sums of kernels possible for george separated by commas)
GP_package      'celerite'  # options: george, celerite

# Manual clipping in time
manual_clip     [[None,500]]   # Recommend removing the first ≥50 integrations that show strong effects of detector settling

# dynesty fitting parameters
run_nlive       'min'         # Must be > ndim * (ndim + 1) // 2. Use 'min' to use the minimum safe number
run_bound       'multi'
run_sample      'rwalk'
run_tol         0.01

# dynamic dynesty fitter (in addition to the dynesty fitter parameters)
run_dynamic     True    # If False, use only dynesty static fitter; if True, use dynamic dynesty fitter
run_nlive_batch 'auto'  # Number of live points per batch for dynamic nested sampling; use int or 'auto' for minimum safe default (max(25, run_nlive // 2)).
run_pfrac       0.5     # Fraction of samples from posterior vs prior during refinement; 0.5 is balanced, higher favors posterior precision, lower improves evidence precision.

# Plotting controls
interp          True    # Should astrophysical model be interpolated (useful for uneven sampling like that from HST)

# Diagnostics
isplots_S5      5       # Generate few (1), some (3), or many (5) figures (Options: 1 - 5)
hideplots       False   # Leave as False to let the images get printed to the notebook

# Project directory
topdir          {topdir}

# Directories relative to topdir
inputdir        {inputdirlist[0]}

inputdirlist    {inputdirlist[1:]}

outputdir       {analysis_name}/Stage5
"""

# This will save the ECF as a file that the next cell can read-in
with open(f'./S5_{eventlabel}.ecf', 'w') as f:
    f.write(s5_ecf_contents)

### 1.2 Setting up the Stage 5 EPF

Make sure to update the astrophysical parameter priors based on those provided in the relevant Jira ticket.

Make sure that the eclipse depth and any not-fixed orbital parameters are set to `'shared'` so that the same value is used for each eclipse. Meanwhile, generally the systematic variables should be left as `'free'` (as there's no strong reason to expect they'd be exactly the same between visits); the one possible exception is the GP covariance lengthscale (`m`) and possible the GP covariance amplitude (`A`), though there's no clear evidence yet that those parameters are the same for all observations.

When adding/removing functions from the `run_myfuncs` list in the ECF above, make sure to add/remove the corresponding parameters from the EPF below.

Some decent initial priors for each of the parameters have been populated below, but you may find that you need to adjust them

If you want to impose different priors between the different eclipses, you can do something like the following:
```
# This would be used for inputdirlist[0] and all other inputdirlist elements not explicitly set below
ypos        0.0                 'free'           0.0           0.5         N
# This would be used for inputdirlist[1]
ypos_ch1    0.0                 'free'           0.0           0.1         N
```

In [ ]:
s5_epf = """
# Stage 5 Fit Parameters Documentation: https://eurekadocs.readthedocs.io/en/latest/ecf.html#stage-5-fit-parameters

#Name         Value                 Free?            PriorPar1        PriorPar2    PriorType
# "Free?" can be free, fixed, white_free, white_fixed, shared, or independent
# PriorType can be U (Uniform), LU (Log Uniform), or N (Normal).
# If U/LU, PriorPar1 and PriorPar2 represent lower and upper limits of the parameter/log(the parameter).
# If N, PriorPar1 is the mean and PriorPar2 is the standard deviation of a Gaussian prior.
#-------------------------------------------------------------------------------------------------------
rp          YourRadiusHere      'fixed'
fp          YourFpHere          'shared'         YourFpHere    2000e-6     N
# ------------------
# Orbital parameters
# ------------------
per         YourPerHere         'fixed'
t_secondary YourTsecHere        'shared'         YourTsecHere  YourTsecUncertHere N
inc         YourIncHere         'fixed'
a           YourAHere           'fixed'
ecc         0.                  'fixed'
w           90.                 'fixed'
# The following two lines are commented out, but you can uncomment them (while commenting out the ecc and w lines above) and edit them if needed for your planet
# ecosw       YourEcoswHere       'fixed'          YourEcoswHere YourEcoswUncertHere N
# esinw       YourEsinwHere       'fixed'          YourEsinwHere YourEsinwUncertHere N
# --------------------------------------------------------------------------
# Systematic variables
# --------------------------------------------------------------------------
# Polynomial Parameters
c0          0.999               'free'           0.999         0.01        N
c1          -0.001              'free'           0.0           0.1         N
# Ramp Parameters
r0          0.001               'free'           0.0           0.01        N
r1          50                  'free'           3             300         U
# Centroid decorrelation parameters
ypos        0.0                 'free'           0.0           0.5         N
xpos        0.0                 'free'           0.0           0.5         N
ywidth      0.0                 'free'           0.0           0.5         N
xwidth      0.0                 'free'           0.0           0.5         N
# --------------------------------------------------------------------------
# Gaussian Process parameters (only used if GP added to s5_meta.run_myfuncs **and** the two parameters below are uncommented).
# Comment these out if GP isn't included in run_myfuncs
# --------------------------------------------------------------------------
# A           -17                 'free'          -24            -10         U
# m           -4                  'free'          -7             0           U
# --------------------------------------------------------------------------
# White noise multiplier
# --------------------------------------------------------------------------
scatter_mult 1.15                 'free'           0.8           10          U
"""

# This code will write your EPF settings to a file and doesn't need to be changed
epf_filename = f'./S5_{eventlabel}.epf'
with open(epf_filename, 'w') as file:
    file.writelines(s5_epf)

### 1.3 Running Stage 5

Here we run the Eureka! Stage 5 pipeline using the settings we defined above.

Note, if you re-run Stage 5 without restarting the Jupyter notebook, the color of your plot will change each time you run Stage 5. This is not an issue and there's no need to restart the notebook; it is just a consequence of the fact that Eureka! was not originally intended to be run within Jupyter notebooks

In [ ]:
s5_meta = s5.fitlc(eventlabel)